# Aggregate Detection Backtesting

This notebook allows you to test the aggregate position detection logic against specific accounts.

## Use Cases
1. **Verify Known Whales**: Test if the detection would catch known insider accounts
2. **Tune Thresholds**: Experiment with different detection parameters
3. **Analyze Patterns**: Understand trading patterns of specific accounts

## Test Account
The primary test case is `0x31a56e9e690c621ed21de08cb559e9524cdb8ed9`:
- Joined Dec 2025
- Made ~$30K+ in trades on "Maduro out by January 31, 2026?" at 7-10%
- Individual trades were all below $10K (evading single-trade detection)
- Profited $400K+ when market resolved

---

## Step 1: Install Dependencies

In [ ]:
!pip install -q requests pandas
print("Dependencies installed!")

## Step 2: Configuration

Set the detection thresholds to test. You can modify these to see how different settings affect detection.

In [ ]:
# =============================================================================
# BACKTEST CONFIGURATION
# =============================================================================

# Account to analyze (the known whale from the example)
TEST_ACCOUNT = "0x31a56e9e690c621ed21de08cb559e9524cdb8ed9"

# Aggregate Detection Thresholds
AGGREGATE_MIN_POSITION_USD = 30000    # Min aggregate position to trigger
ASYMMETRIC_PRICE_THRESHOLD = 0.30     # Price threshold (30% = low-odds)
MAX_SINGLE_TRADE_FOR_AGGREGATE = 10000  # Max single trade for stealth pattern

# Fresh Account Thresholds
MAX_HISTORICAL_TRADES = 5   # Max trades for "new account"
NEW_ACCOUNT_HOURS = 72 * 24  # Extended to 72 days for backtesting

# Lookback period (days)
LOOKBACK_DAYS = 30

print("Configuration loaded!")
print(f"Test Account: {TEST_ACCOUNT}")
print(f"\nDetection Thresholds:")
print(f"  - Min aggregate position: ${AGGREGATE_MIN_POSITION_USD:,}")
print(f"  - Asymmetric threshold: {ASYMMETRIC_PRICE_THRESHOLD:.0%}")
print(f"  - Max single trade (stealth): ${MAX_SINGLE_TRADE_FOR_AGGREGATE:,}")
print(f"  - Lookback: {LOOKBACK_DAYS} days")

## Step 3: Load Core Detection Code

In [ ]:
import time
import requests
import pandas as pd
from datetime import datetime, timezone, timedelta
from typing import Optional, Dict, List, Any
from dataclasses import dataclass
from IPython.display import display, HTML

# =============================================================================
# DATA MODELS
# =============================================================================

@dataclass
class MarketPosition:
    """Aggregated position in a single market."""
    market_id: str
    market_title: str
    outcome: str
    total_shares: float
    average_price: float
    total_invested_usd: float
    trade_count: int
    first_trade_timestamp: int
    last_trade_timestamp: int
    max_single_trade_usd: float
    trades: List[Dict]  # Store individual trades for analysis

    @property
    def time_span_hours(self) -> float:
        if self.first_trade_timestamp and self.last_trade_timestamp:
            return (self.last_trade_timestamp - self.first_trade_timestamp) / 3600
        return 0


@dataclass 
class AccountProfile:
    """Profile information for a Polymarket account."""
    address: str
    total_trades: int
    first_trade_timestamp: Optional[int]
    total_volume_usd: float
    markets_traded: int

    @property
    def account_age_hours(self) -> Optional[float]:
        if not self.first_trade_timestamp:
            return None
        age_seconds = time.time() - self.first_trade_timestamp
        return age_seconds / 3600


# =============================================================================
# SUBGRAPH CLIENT
# =============================================================================

class PolymarketSubgraph:
    """Client for querying Polymarket data."""
    
    SUBGRAPH_URL = (
        "https://api.goldsky.com/api/public/"
        "project_cl6mb8i9h0003e201j6li0diw/subgraphs/polymarket-subgraph/prod/gn"
    )
    
    def __init__(self):
        self.session = requests.Session()
    
    def _execute_query(self, query: str, variables: Dict = None) -> Optional[Dict]:
        payload = {"query": query}
        if variables:
            payload["variables"] = variables
        
        for attempt in range(3):
            try:
                response = self.session.post(
                    self.SUBGRAPH_URL,
                    json=payload,
                    timeout=30,
                    headers={"Content-Type": "application/json"}
                )
                response.raise_for_status()
                result = response.json()
                if "errors" in result:
                    print(f"GraphQL errors: {result['errors']}")
                    return None
                return result.get("data")
            except Exception as e:
                print(f"Request failed (attempt {attempt + 1}): {e}")
                if attempt < 2:
                    time.sleep(2)
        return None

    def get_account_all_trades(self, address: str, first: int = 1000) -> List[Dict]:
        """Fetch ALL trades for an account (for comprehensive analysis)."""
        query = """
        query GetAccountTrades($address: String!, $first: Int!) {
            user(id: $address) {
                id
                trades(first: $first, orderBy: timestamp, orderDirection: asc) {
                    id
                    market { id question }
                    outcome
                    amount
                    price
                    timestamp
                    transactionHash
                }
            }
        }
        """
        
        alt_query = """
        query GetAccountTrades($address: String!, $first: Int!) {
            account(id: $address) {
                id
                fpmmTrades(first: $first, orderBy: creationTimestamp, orderDirection: asc) {
                    id
                    fpmm { id question }
                    outcomeIndex
                    collateralAmount
                    outcomeTokensAmount
                    creationTimestamp
                    transactionHash
                }
            }
        }
        """
        
        variables = {"address": address.lower(), "first": first}
        data = self._execute_query(query, variables)
        
        trades = []
        
        if data and data.get("user"):
            for t in data["user"].get("trades", []):
                try:
                    amount = float(t.get("amount", 0))
                    price = float(t.get("price", 0))
                    trades.append({
                        "id": t["id"],
                        "market_id": t["market"]["id"],
                        "market_title": t["market"].get("question", "Unknown"),
                        "outcome": t.get("outcome", "Unknown"),
                        "amount": amount,
                        "price": price,
                        "value_usd": amount * price,
                        "timestamp": int(t["timestamp"]),
                        "tx_hash": t.get("transactionHash", "")
                    })
                except:
                    continue
            return trades
        
        # Try alternative schema
        data = self._execute_query(alt_query, variables)
        if data and data.get("account"):
            for t in data["account"].get("fpmmTrades", []):
                try:
                    collateral = float(t.get("collateralAmount", 0)) / 1e6
                    outcome_tokens = float(t.get("outcomeTokensAmount", 0)) / 1e18
                    price = collateral / outcome_tokens if outcome_tokens > 0 else 0
                    outcome = "Yes" if int(t.get("outcomeIndex", 0)) == 0 else "No"
                    trades.append({
                        "id": t["id"],
                        "market_id": t["fpmm"]["id"],
                        "market_title": t["fpmm"].get("question", "Unknown"),
                        "outcome": outcome,
                        "amount": outcome_tokens,
                        "price": price,
                        "value_usd": collateral,
                        "timestamp": int(t["creationTimestamp"]),
                        "tx_hash": t.get("transactionHash", "")
                    })
                except:
                    continue
        
        return trades

    def get_account_profile(self, address: str) -> AccountProfile:
        """Get account profile summary."""
        trades = self.get_account_all_trades(address)
        
        if not trades:
            return AccountProfile(
                address=address,
                total_trades=0,
                first_trade_timestamp=None,
                total_volume_usd=0,
                markets_traded=0
            )
        
        return AccountProfile(
            address=address,
            total_trades=len(trades),
            first_trade_timestamp=trades[0]["timestamp"] if trades else None,
            total_volume_usd=sum(t["value_usd"] for t in trades),
            markets_traded=len(set(t["market_id"] for t in trades))
        )


def aggregate_trades_to_positions(trades: List[Dict]) -> Dict[str, MarketPosition]:
    """Aggregate trades into market positions."""
    position_data: Dict[str, Dict] = {}
    
    for t in trades:
        key = f"{t['market_id']}:{t['outcome']}"
        
        if key not in position_data:
            position_data[key] = {
                "market_id": t["market_id"],
                "market_title": t["market_title"],
                "outcome": t["outcome"],
                "total_shares": 0,
                "total_invested_usd": 0,
                "trade_count": 0,
                "first_trade_timestamp": t["timestamp"],
                "last_trade_timestamp": t["timestamp"],
                "max_single_trade_usd": 0,
                "prices": [],
                "trades": []
            }
        
        pos = position_data[key]
        pos["total_shares"] += t["amount"]
        pos["total_invested_usd"] += t["value_usd"]
        pos["trade_count"] += 1
        pos["last_trade_timestamp"] = max(pos["last_trade_timestamp"], t["timestamp"])
        pos["first_trade_timestamp"] = min(pos["first_trade_timestamp"], t["timestamp"])
        pos["max_single_trade_usd"] = max(pos["max_single_trade_usd"], t["value_usd"])
        pos["prices"].append((t["value_usd"], t["price"]))
        pos["trades"].append(t)
    
    # Convert to MarketPosition objects
    positions: Dict[str, MarketPosition] = {}
    for key, pos in position_data.items():
        total_value = sum(p[0] for p in pos["prices"])
        avg_price = sum(p[0] * p[1] for p in pos["prices"]) / total_value if total_value > 0 else 0
        
        positions[key] = MarketPosition(
            market_id=pos["market_id"],
            market_title=pos["market_title"],
            outcome=pos["outcome"],
            total_shares=pos["total_shares"],
            average_price=avg_price,
            total_invested_usd=pos["total_invested_usd"],
            trade_count=pos["trade_count"],
            first_trade_timestamp=pos["first_trade_timestamp"],
            last_trade_timestamp=pos["last_trade_timestamp"],
            max_single_trade_usd=pos["max_single_trade_usd"],
            trades=pos["trades"]
        )
    
    return positions


print("Detection code loaded!")

## Step 4: Fetch Account Data

This will fetch all trades for the test account and display them.

In [ ]:
# Initialize subgraph client
subgraph = PolymarketSubgraph()

print(f"Fetching data for account: {TEST_ACCOUNT}")
print("=" * 60)

# Get account profile
profile = subgraph.get_account_profile(TEST_ACCOUNT)

print(f"\nAccount Profile:")
print(f"  Total trades: {profile.total_trades}")
print(f"  Total volume: ${profile.total_volume_usd:,.2f}")
print(f"  Markets traded: {profile.markets_traded}")
if profile.first_trade_timestamp:
    first_trade_date = datetime.fromtimestamp(profile.first_trade_timestamp, tz=timezone.utc)
    print(f"  First trade: {first_trade_date.strftime('%Y-%m-%d %H:%M UTC')}")
    if profile.account_age_hours:
        print(f"  Account age: {profile.account_age_hours / 24:.1f} days")

# Get all trades
all_trades = subgraph.get_account_all_trades(TEST_ACCOUNT)
print(f"\nFetched {len(all_trades)} trades")

## Step 5: View All Trades

Display all trades in a table format.

In [ ]:
# Convert to DataFrame for display
if all_trades:
    df_trades = pd.DataFrame([
        {
            "Date": datetime.fromtimestamp(t["timestamp"], tz=timezone.utc).strftime("%Y-%m-%d %H:%M"),
            "Market": t["market_title"][:50] + "..." if len(t["market_title"]) > 50 else t["market_title"],
            "Outcome": t["outcome"],
            "Price": f"{t['price']:.2%}",
            "Shares": f"{t['amount']:,.2f}",
            "Value (USD)": f"${t['value_usd']:,.2f}",
        }
        for t in all_trades
    ])
    
    print(f"All Trades for {TEST_ACCOUNT[:10]}...{TEST_ACCOUNT[-6:]}")
    print("=" * 100)
    display(df_trades)
else:
    print("No trades found for this account.")

## Step 6: Aggregate into Positions

Group trades by market to see the aggregate positions.

In [ ]:
# Aggregate trades into positions
positions = aggregate_trades_to_positions(all_trades)

print(f"\nAggregated Positions ({len(positions)} total)")
print("=" * 100)

# Convert to DataFrame
if positions:
    df_positions = pd.DataFrame([
        {
            "Market": pos.market_title[:40] + "..." if len(pos.market_title) > 40 else pos.market_title,
            "Outcome": pos.outcome,
            "Avg Price": f"{pos.average_price:.2%}",
            "# Trades": pos.trade_count,
            "Total Shares": f"{pos.total_shares:,.2f}",
            "Total Invested": f"${pos.total_invested_usd:,.2f}",
            "Max Single Trade": f"${pos.max_single_trade_usd:,.2f}",
            "Time Span": f"{pos.time_span_hours / 24:.1f} days"
        }
        for pos in sorted(positions.values(), key=lambda x: x.total_invested_usd, reverse=True)
    ])
    display(df_positions)
else:
    print("No positions found.")

## Step 7: Run Aggregate Detection

Apply the detection criteria to each position and see which ones would trigger an alert.

In [ ]:
def check_aggregate_detection(position: MarketPosition, profile: AccountProfile) -> Dict:
    """Check if a position meets aggregate whale detection criteria."""
    checks = {
        "position": position,
        "checks": [],
        "passed": True,
        "would_alert": False
    }
    
    # Check 1: Aggregate value threshold
    agg_check = position.total_invested_usd >= AGGREGATE_MIN_POSITION_USD
    checks["checks"].append({
        "name": "Aggregate Value",
        "condition": f">= ${AGGREGATE_MIN_POSITION_USD:,}",
        "actual": f"${position.total_invested_usd:,.2f}",
        "passed": agg_check
    })
    if not agg_check:
        checks["passed"] = False
    
    # Check 2: Asymmetric market (low odds)
    asym_check = position.average_price < ASYMMETRIC_PRICE_THRESHOLD
    checks["checks"].append({
        "name": "Asymmetric Market",
        "condition": f"< {ASYMMETRIC_PRICE_THRESHOLD:.0%}",
        "actual": f"{position.average_price:.2%}",
        "passed": asym_check
    })
    if not asym_check:
        checks["passed"] = False
    
    # Check 3: Stealth pattern (no large single trade)
    stealth_check = position.max_single_trade_usd < MAX_SINGLE_TRADE_FOR_AGGREGATE
    checks["checks"].append({
        "name": "Stealth Pattern",
        "condition": f"max single < ${MAX_SINGLE_TRADE_FOR_AGGREGATE:,}",
        "actual": f"${position.max_single_trade_usd:,.2f}",
        "passed": stealth_check
    })
    if not stealth_check:
        checks["passed"] = False
    
    # Check 4: Multiple trades
    multi_check = position.trade_count >= 2
    checks["checks"].append({
        "name": "Multiple Trades",
        "condition": ">= 2",
        "actual": str(position.trade_count),
        "passed": multi_check
    })
    if not multi_check:
        checks["passed"] = False
    
    # Check 5: Fresh account
    fresh_by_trades = profile.total_trades < MAX_HISTORICAL_TRADES
    fresh_by_age = (
        profile.account_age_hours is not None and 
        profile.account_age_hours < NEW_ACCOUNT_HOURS
    )
    fresh_check = fresh_by_trades or fresh_by_age
    
    fresh_reason = []
    if fresh_by_trades:
        fresh_reason.append(f"trades={profile.total_trades}<{MAX_HISTORICAL_TRADES}")
    if fresh_by_age:
        fresh_reason.append(f"age={profile.account_age_hours/24:.0f}d<{NEW_ACCOUNT_HOURS/24:.0f}d")
    
    checks["checks"].append({
        "name": "Fresh Account",
        "condition": f"<{MAX_HISTORICAL_TRADES} trades OR <{NEW_ACCOUNT_HOURS/24:.0f} days old",
        "actual": ", ".join(fresh_reason) if fresh_reason else f"trades={profile.total_trades}, age={profile.account_age_hours/24:.0f}d",
        "passed": fresh_check
    })
    if not fresh_check:
        checks["passed"] = False
    
    checks["would_alert"] = checks["passed"]
    
    return checks


# Run detection on all positions
print("AGGREGATE DETECTION RESULTS")
print("=" * 100)

detection_results = []
alerts_triggered = []

for key, position in positions.items():
    result = check_aggregate_detection(position, profile)
    detection_results.append(result)
    
    if result["would_alert"]:
        alerts_triggered.append(result)

# Summary
print(f"\nTotal positions analyzed: {len(positions)}")
print(f"Positions that would trigger alert: {len(alerts_triggered)}")

if alerts_triggered:
    print(f"\n{'='*100}")
    print("ALERTS WOULD BE TRIGGERED FOR:")
    print("="*100)
    
    for result in alerts_triggered:
        pos = result["position"]
        print(f"\n Market: {pos.market_title}")
        print(f"   Position: {pos.outcome} @ {pos.average_price:.2%}")
        print(f"   Total Invested: ${pos.total_invested_usd:,.2f}")
        print(f"   Trades: {pos.trade_count}")
        print(f"   Max Single: ${pos.max_single_trade_usd:,.2f}")
        print(f"   Time Span: {pos.time_span_hours/24:.1f} days")
        print(f"   DETECTION REASON: Accumulated ${pos.total_invested_usd:,.0f} through {pos.trade_count} trades at avg {pos.average_price:.1%}")

## Step 8: Detailed Check Results

See the detailed pass/fail status for each detection criterion on each position.

In [ ]:
# Detailed results for all positions
print("DETAILED DETECTION CHECKS")
print("=" * 100)

for result in sorted(detection_results, key=lambda x: x["position"].total_invested_usd, reverse=True):
    pos = result["position"]
    alert_status = "WOULD ALERT" if result["would_alert"] else "NO ALERT"
    color = "#00ff00" if result["would_alert"] else "#ff6666"
    
    print(f"\n{'='*80}")
    print(f"Market: {pos.market_title[:60]}...")
    print(f"Position: {pos.outcome} @ {pos.average_price:.2%} | ${pos.total_invested_usd:,.2f} invested")
    print(f"Status: [{alert_status}]")
    print("-" * 80)
    
    for check in result["checks"]:
        status = "PASS" if check["passed"] else "FAIL"
        print(f"  [{status}] {check['name']}: {check['actual']} (need: {check['condition']})")

## Step 9: Individual Trade Breakdown

For positions that would trigger alerts, show the individual trades.

In [ ]:
if alerts_triggered:
    for result in alerts_triggered:
        pos = result["position"]
        print(f"\n{'='*100}")
        print(f"TRADE BREAKDOWN: {pos.market_title[:60]}...")
        print(f"Position: {pos.outcome} | Avg Price: {pos.average_price:.2%}")
        print("="*100)
        
        df_breakdown = pd.DataFrame([
            {
                "#": i + 1,
                "Date": datetime.fromtimestamp(t["timestamp"], tz=timezone.utc).strftime("%Y-%m-%d %H:%M"),
                "Price": f"{t['price']:.2%}",
                "Shares": f"{t['amount']:,.2f}",
                "Value (USD)": f"${t['value_usd']:,.2f}",
                "Running Total": f"${sum(x['value_usd'] for x in pos.trades[:i+1]):,.2f}"
            }
            for i, t in enumerate(pos.trades)
        ])
        display(df_breakdown)
        
        print(f"\nSUMMARY:")
        print(f"  Total trades: {len(pos.trades)}")
        print(f"  Total invested: ${pos.total_invested_usd:,.2f}")
        print(f"  Total shares: {pos.total_shares:,.2f}")
        print(f"  Avg price: {pos.average_price:.2%}")
        print(f"  Max single trade: ${pos.max_single_trade_usd:,.2f}")
        print(f"  Potential payout (if wins): ${pos.total_shares:,.2f}")
        print(f"  Potential profit: ${pos.total_shares - pos.total_invested_usd:,.2f}")
else:
    print("No alerts triggered - nothing to break down.")
    print("\nTry adjusting the detection thresholds in Step 2 and re-running.")

## Step 10: Test Multiple Accounts

You can test additional accounts by modifying the list below and running this cell.

In [ ]:
# Add additional accounts to test here
ADDITIONAL_ACCOUNTS = [
    # "0x...",  # Add more accounts to test
]

if ADDITIONAL_ACCOUNTS:
    print("Testing additional accounts...")
    print("=" * 100)
    
    for account in ADDITIONAL_ACCOUNTS:
        print(f"\n\nAnalyzing: {account}")
        print("-" * 60)
        
        # Get data
        acc_profile = subgraph.get_account_profile(account)
        acc_trades = subgraph.get_account_all_trades(account)
        acc_positions = aggregate_trades_to_positions(acc_trades)
        
        print(f"Total trades: {acc_profile.total_trades}")
        print(f"Total volume: ${acc_profile.total_volume_usd:,.2f}")
        print(f"Positions: {len(acc_positions)}")
        
        # Check each position
        alerts = []
        for key, pos in acc_positions.items():
            result = check_aggregate_detection(pos, acc_profile)
            if result["would_alert"]:
                alerts.append(result)
        
        if alerts:
            print(f"\n*** {len(alerts)} ALERT(S) WOULD TRIGGER ***")
            for r in alerts:
                p = r["position"]
                print(f"  - {p.market_title[:40]}... | {p.outcome} @ {p.average_price:.1%} | ${p.total_invested_usd:,.0f}")
        else:
            print("No alerts would trigger.")
else:
    print("No additional accounts to test.")
    print("Add account addresses to ADDITIONAL_ACCOUNTS list above.")

---

## Summary

This notebook allows you to:

1. **Fetch all trades** for any Polymarket account
2. **View positions** aggregated by market
3. **Run detection criteria** against each position
4. **See detailed pass/fail** for each criterion
5. **View trade-by-trade breakdown** for flagged positions
6. **Test multiple accounts** at once

### Adjusting Detection Sensitivity

To catch more or fewer accounts, modify these values in Step 2:

| Parameter | Lower = More Sensitive | Higher = Less Sensitive |
|-----------|------------------------|------------------------|
| `AGGREGATE_MIN_POSITION_USD` | $20K catches smaller positions | $50K catches only large positions |
| `ASYMMETRIC_PRICE_THRESHOLD` | 0.20 (20%) = very low odds only | 0.40 (40%) = includes moderate odds |
| `MAX_SINGLE_TRADE_FOR_AGGREGATE` | $5K = stricter stealth requirement | $15K = allows larger individual trades |

---